# Infectious Disease Data Analysis with PostgreSQL

## Завантаження даних

Завантажуємо CSV у Colab і перевіряємо розмір таблиці, типи стовпців та пропущені значення.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

csv_filename = next(iter(uploaded))
df = pd.read_csv(csv_filename)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

Saving infectious_cases.csv to infectious_cases.csv
Dataset shape: (10521, 12)

Columns:
['Entity', 'Code', 'Year', 'Number_yaws', 'polio_cases', 'cases_guinea_worm', 'Number_rabies', 'Number_malaria', 'Number_hiv', 'Number_tuberculosis', 'Number_smallpox', 'Number_cholera_cases']

First 5 rows:


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Data types:
Entity                   object
Code                     object
Year                      int64
Number_yaws             float64
polio_cases             float64
cases_guinea_worm       float64
Number_rabies           float64
Number_malaria          float64
Number_hiv              float64
Number_tuberculosis     float64
Number_smallpox         float64
Number_cholera_cases    float64
dtype: object

Missing values:
Entity                      0
Code                     1128
Year                        0
Number_yaws             10447
polio_cases              2911
cases_guinea_worm        2093
Number_rabies            3741
Number_malaria           3741
Number_hiv               3741
Number_tuberculosis      3741
Number_smallpox         10417
Number_cholera_cases     7554
dtype: int64


## Підключення PostgreSQL

Встановлюємо бібліотеки й запускаємо PostgreSQL у середовищі Colab.

In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 83.5 MB/s eta 0:00:00


In [ ]:
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## Імпорт у базу

Створюємо схему `pandemic`, переносимо дані в таблицю `infectious_cases` і перевіряємо кількість рядків. Повторний запуск імпорту замінює вихідну таблицю.

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS pandemic"))
    conn.execute(text("SET search_path TO pandemic"))

print("Schema pandemic created")

Schema pandemic created


In [ ]:
df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=1000
)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM pandemic.infectious_cases")
    ).scalar_one()

print("Rows imported:", count)

Rows imported: 10521


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT *
        FROM pandemic.infectious_cases
        LIMIT 5
    """))

    for row in result:
        print(row)

('Afghanistan', 'AFG', 1980, None, 6160.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1981, None, 5859.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1982, None, 9730.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1983, None, 13937.0, 0.0, None, None, None, None, None, None)
('Afghanistan', 'AFG', 1984, None, 3864.0, 0.0, None, None, None, None, None, None)


## Нормалізація

Перевіряємо кількість унікальних пар `Entity`–`Code` і виносимо їх в окремий довідник `entities`.

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT ("Entity", "Code")) AS unique_pairs
        FROM pandemic.infectious_cases
    """))

    print(result.fetchone())

(10521, 245)


In [ ]:
create_entities_sql = """
DROP TABLE IF EXISTS pandemic.infectious_cases_normalized;
DROP TABLE IF EXISTS pandemic.entities;

CREATE TABLE pandemic.entities (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE (entity, code)
);

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT
    "Entity",
    "Code"
FROM pandemic.infectious_cases;
"""

with engine.begin() as conn:
    conn.execute(text(create_entities_sql))

245


TypeError: read_sql() missing 1 required positional argument: 'con'

In [ ]:
with engine.connect() as conn:
    entities_df = pd.read_sql(
        """
        SELECT *
        FROM pandemic.entities
        ORDER BY id
        LIMIT 10
        """,
        con=conn
    )

display(entities_df)

,id,entity,code
0,1,Russia,RUS
1,2,Trinidad and Tobago,TTO
2,3,Antigua and Barbuda,ATG
3,4,Nicaragua,NIC
4,5,Denmark,DNK
5,6,Turkey,TUR
6,7,Sudan,SDN
7,8,Myanmar,MMR
8,9,Macao,MAC
9,10,Togo,TGO


Річні показники зберігаємо в `infectious_cases_normalized` із посиланням на `entities`. Для зіставлення пропущених кодів використовуємо `IS NOT DISTINCT FROM`.

In [ ]:
create_normalized_sql = """
CREATE TABLE pandemic.infectious_cases_normalized (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_id INTEGER NOT NULL,
    year INTEGER NOT NULL,
    number_yaws DOUBLE PRECISION,
    polio_cases DOUBLE PRECISION,
    cases_guinea_worm DOUBLE PRECISION,
    number_rabies DOUBLE PRECISION,
    number_malaria DOUBLE PRECISION,
    number_hiv DOUBLE PRECISION,
    number_tuberculosis DOUBLE PRECISION,
    number_smallpox DOUBLE PRECISION,
    number_cholera_cases DOUBLE PRECISION,

    FOREIGN KEY (entity_id)
        REFERENCES pandemic.entities(id)
);

INSERT INTO pandemic.infectious_cases_normalized (
    entity_id,
    year,
    number_yaws,
    polio_cases,
    cases_guinea_worm,
    number_rabies,
    number_malaria,
    number_hiv,
    number_tuberculosis,
    number_smallpox,
    number_cholera_cases
)
SELECT
    e.id,
    ic."Year",
    ic."Number_yaws",
    ic."polio_cases",
    ic."cases_guinea_worm",
    ic."Number_rabies",
    ic."Number_malaria",
    ic."Number_hiv",
    ic."Number_tuberculosis",
    ic."Number_smallpox",
    ic."Number_cholera_cases"
FROM pandemic.infectious_cases AS ic
JOIN pandemic.entities AS e
    ON ic."Entity" = e.entity
   AND ic."Code" IS NOT DISTINCT FROM e.code;
"""

with engine.begin() as conn:
    conn.execute(text(create_normalized_sql))

print("Normalized rows:")
with engine.connect() as conn:
    print(conn.execute(text("""
        SELECT COUNT(*)
        FROM pandemic.infectious_cases_normalized
    """)).scalar_one())

    display(pd.read_sql("""
        SELECT *
        FROM pandemic.infectious_cases_normalized
        ORDER BY id
        LIMIT 5
    """, con=conn))

Normalized rows:
10521


,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,212,1980,None,6160.0,0.0,None,None,None,None,None,None
1,2,212,1981,None,5859.0,0.0,None,None,None,None,None,None
2,3,212,1982,None,9730.0,0.0,None,None,None,None,None,None
3,4,212,1983,None,13937.0,0.0,None,None,None,None,None,None
4,5,212,1984,None,3864.0,0.0,None,None,None,None,None,None


## Перевірка зв’язків

Порівнюємо кількість рядків до й після нормалізації та перевіряємо, чи немає записів без відповідної сутності в довіднику.

In [ ]:
with engine.connect() as conn:
    checks = pd.read_sql("""
        SELECT
            (SELECT COUNT(*)
             FROM pandemic.infectious_cases) AS original_rows,

            (SELECT COUNT(*)
             FROM pandemic.infectious_cases_normalized) AS normalized_rows,

            (SELECT COUNT(*)
             FROM pandemic.entities) AS entities_count,

            (SELECT COUNT(*)
             FROM pandemic.infectious_cases_normalized n
             LEFT JOIN pandemic.entities e
                ON n.entity_id = e.id
             WHERE e.id IS NULL) AS orphan_rows
    """, con=conn)

display(checks)

,original_rows,normalized_rows,entities_count,orphan_rows
0,10521,10521,245,0


## Показники сказу

Обчислюємо середнє, мінімум, максимум і суму за кожною сутністю та виводимо топ-10 за середнім. У вибірці є країни й агреговані групи; пропуски не замінюємо нулями.

In [ ]:
rabies_sql = """
SELECT
    e.entity,
    e.code,
    AVG(n.number_rabies) AS average_rabies,
    MIN(n.number_rabies) AS min_rabies,
    MAX(n.number_rabies) AS max_rabies,
    SUM(n.number_rabies) AS total_rabies
FROM pandemic.infectious_cases_normalized AS n
JOIN pandemic.entities AS e
    ON n.entity_id = e.id
GROUP BY e.id, e.entity, e.code
HAVING COUNT(n.number_rabies) > 0
ORDER BY average_rabies DESC
LIMIT 10;
"""

with engine.connect() as conn:
    rabies_df = pd.read_sql(rabies_sql, con=conn)

display(rabies_df)


,entity,code,average_rabies,min_rabies,max_rabies,total_rabies
0,World,OWID_WRL,20192.370367,14075.5080,24744.6580,605771.1110
1,Lower Middle Income (WB),None,15193.959383,10202.5280,19182.7950,455818.7815
2,South Asia (WB),None,11729.888753,7271.2754,15361.8780,351896.6626
3,South-East Asia Region (WHO),None,11424.326893,6806.0127,15641.9580,342729.8068
4,G20,None,10189.046003,6339.0796,13164.8810,305671.3801
5,India,IND,8599.173280,5425.8726,11121.1420,257975.1984
6,Sub-Saharan Africa (WB),None,4748.268090,4356.8184,5017.9770,142448.0427
7,African Region (WHO),None,4649.702377,4283.0730,4917.7840,139491.0713
8,Low Income (WB),None,3568.105450,3014.6170,3873.5330,107043.1635
9,East Asia & Pacific (WB),None,3427.461133,2009.9395,4591.0107,102823.8340


## Робота з датами

Для кожного року створюємо дату 1 січня й визначаємо кількість повних років до поточної дати.

In [ ]:
year_diff_sql = """
SELECT
    entity_id,
    year,
    MAKE_DATE(year, 1, 1) AS year_start,
    CURRENT_DATE AS current_date,
    EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(year, 1, 1)
        )
    ) AS year_difference
FROM pandemic.infectious_cases_normalized
ORDER BY year
LIMIT 10;
"""

with engine.connect() as conn:
    year_diff_df = pd.read_sql(year_diff_sql, con=conn)

display(year_diff_df)

,entity_id,year,year_start,current_date,year_difference
0,84,1920,1920-01-01,2026-10-17,106.0
1,84,1921,1921-01-01,2026-10-17,105.0
2,84,1922,1922-01-01,2026-10-17,104.0
3,84,1923,1923-01-01,2026-10-17,103.0
4,84,1924,1924-01-01,2026-10-17,102.0
5,84,1925,1925-01-01,2026-10-17,101.0
6,84,1926,1926-01-01,2026-10-17,100.0
7,84,1927,1927-01-01,2026-10-17,99.0
8,84,1928,1928-01-01,2026-10-17,98.0
9,84,1929,1929-01-01,2026-10-17,97.0


## Функція різниці років

Виносимо розрахунок у `year_difference()` і перевіряємо її на окремому значенні та даних таблиці.

In [ ]:
create_year_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.year_difference(input_year INTEGER)
RETURNS INTEGER
LANGUAGE SQL
AS $$
    SELECT EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(input_year, 1, 1)
        )
    )::INTEGER;
$$;
"""

with engine.begin() as conn:
    conn.execute(text(create_year_function_sql))

print("Function year_difference created")

Function year_difference created


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            1996 AS input_year,
            pandemic.year_difference(1996) AS year_difference
    """))

    print(result.fetchone())

(1996, 30)


In [ ]:
with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT
            year,
            pandemic.year_difference(year) AS year_difference
        FROM pandemic.infectious_cases_normalized
        ORDER BY year
        LIMIT 10;
    """), con=conn)

display(result)

,year,year_difference
0,1920,106
1,1921,105
2,1922,104
3,1923,103
4,1924,102
5,1925,101
6,1926,100
7,1927,99
8,1928,98
9,1929,97


## Середнє за період

Функція `average_cases()` ділить річний показник на задану кількість періодів. Для нульового дільника або пропущених аргументів повертає `NULL`.

In [ ]:
create_average_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.average_cases(
    cases_value DOUBLE PRECISION,
    divisor DOUBLE PRECISION
)
RETURNS DOUBLE PRECISION
LANGUAGE SQL
IMMUTABLE
AS $$
    SELECT CASE
        WHEN cases_value IS NULL OR divisor IS NULL OR divisor = 0
            THEN NULL
        ELSE cases_value / divisor
    END;
$$;
"""

with engine.begin() as conn:
    conn.execute(text(create_average_function_sql))

print("Function average_cases created")

Function average_cases created


Демонструємо ділення на 12, 4 і 2. Це середні значення за період, а не фактичні місячні чи квартальні спостереження.

In [ ]:
with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT
            entity_id,
            year,
            number_rabies,
            pandemic.average_cases(number_rabies, 12) AS monthly_avg,
            pandemic.average_cases(number_rabies, 4) AS quarterly_avg,
            pandemic.average_cases(number_rabies, 2) AS half_year_avg
        FROM pandemic.infectious_cases_normalized
        WHERE number_rabies IS NOT NULL
        ORDER BY entity_id, year
        LIMIT 10;
    """), con=conn)

display(result)

,entity_id,year,number_rabies,monthly_avg,quarterly_avg,half_year_avg
0,1,1990,9.106218,0.758852,2.276555,4.553109
1,1,1991,8.998902,0.749908,2.249725,4.499451
2,1,1992,8.946273,0.745523,2.236568,4.473136
3,1,1993,8.923228,0.743602,2.230807,4.461614
4,1,1994,8.930097,0.744175,2.232524,4.465048
5,1,1995,8.992681,0.749390,2.248170,4.496340
6,1,1996,9.169190,0.764099,2.292298,4.584595
7,1,1997,9.464535,0.788711,2.366134,4.732267
8,1,1998,9.810057,0.817505,2.452514,4.905029
9,1,1999,10.123583,0.843632,2.530896,5.061791


In [ ]:
with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT
            year,
            pandemic.year_difference(year) AS year_difference,
            number_rabies,
            pandemic.average_cases(number_rabies, 12) AS monthly_avg,
            pandemic.average_cases(number_rabies, 4) AS quarterly_avg,
            pandemic.average_cases(number_rabies, 2) AS half_year_avg
        FROM pandemic.infectious_cases_normalized
        WHERE number_rabies IS NOT NULL
        ORDER BY year DESC
        LIMIT 10;
    """), con=conn)

display(result)

,year,year_difference,number_rabies,monthly_avg,quarterly_avg,half_year_avg
0,2019,7,0.000138,0.000012,0.000035,0.000069
1,2019,7,0.567866,0.047322,0.141967,0.283933
2,2019,7,0.007448,0.000621,0.001862,0.003724
3,2019,7,4378.748000,364.895667,1094.687000,2189.374000
4,2019,7,7.964379,0.663698,1.991095,3.982190
5,2019,7,0.004845,0.000404,0.001211,0.002422
6,2019,7,1.346161,0.112180,0.336540,0.673080
7,2019,7,0.014225,0.001185,0.003556,0.007112
8,2019,7,3.387213,0.282268,0.846803,1.693607
9,2019,7,0.369019,0.030752,0.092255,0.184509
